# TransUNet 3D Inference with TTA & Post-Processing

Inference notebook for the Vesuvius Challenge - Surface Detection competition.  
Generates submission files using a single TransUNet model combined with TTA and topology-aware post-processing.

## Task
Segment the recto (front) surface of ancient papyrus scrolls from 3D CT scan volumes.  
Input: 3D TIFF volume → Output: binary 3D mask → ZIP submission

## Pipeline

```mermaid
graph LR
    A[Input Volume .tif] --> B[NormalizeIntensity<br/>z-score, nonzero]
    B --> C[TransUNet-SEResNeXt50<br/>160px, comboloss<br/>LB 0.545]
    C --> D[Sliding Window Inference<br/>Gaussian, overlap=0.40]
    D --> E[TTA x8<br/>3 flips + 3 rotations]
    E --> F[Logit Averaging<br/>→ Argmax]
    F --> G[Hysteresis Threshold<br/>T_high=0.80, T_low=0.30]
    G --> H[Anisotropic Closing<br/>z=3, xy=2]
    H --> I[Dust Removal<br/>min_size=100]
    I --> J[Output Mask .tif<br/>→ submission.zip]
```

## Design Choices
- Single model (comboloss, LB 0.545) used — performed better than multi-model ensembles
- TTA logits averaged before argmax to preserve class boundaries
- Aggressive morphological closing (z=3, xy=2) to fill gaps in thin sheet predictions
- Sliding window overlap tuned to 0.40

### Experiment Results
| Configuration | LB Score | Notes |
|---|---|---|
| Single model (comboloss) + TTA + PP (this notebook) | **Best** | overlap=0.40, z=3, xy=2 |
| 2-model ensemble (0.25:0.75) | 3rd | overlap=0.50, z=1, xy=0 |
| Single model (softmax) + TTA + PP | 2nd | overlap=0.35, z=3, xy=2 |

---

## 0. Setup & Dependencies

In [ ]:
from IPython.display import clear_output

var = "/kaggle/input/vsdetection-packages-offline-installer-only/whls"
!pip install \
  "$var"/keras_nightly-*.whl \
  "$var"/tifffile-*.whl \
  "$var"/imagecodecs-*.whl \
  "$var"/medicai-*.whl \
  --no-index \
  --find-links "$var"

clear_output()

In [ ]:
import os
os.environ["KERAS_BACKEND"] = "jax"

import keras
from medicai.transforms import (
    Compose,
    ScaleIntensityRange,
    NormalizeIntensity
)
from medicai.models import SegFormer, TransUNet
from medicai.utils.inference import SlidingWindowInference

import numpy as np
import pandas as pd
import zipfile
import tifffile
import scipy.ndimage as ndi
from skimage.morphology import remove_small_objects
from matplotlib import pyplot as plt

# Reproducibility
SEED = 42
np.random.seed(SEED)

keras.config.backend(), keras.version()

## 1. Configuration

All paths and hyperparameters are centralized here for easy tuning.

In [ ]:
class CFG:
    # --- Paths ---
    root_dir = "/kaggle/input/vesuvius-challenge-surface-detection"
    test_dir = f"{root_dir}/test_images"
    output_dir = "/kaggle/working/submission_masks"
    zip_path = "/kaggle/working/submission.zip"
    
    # Model weights directory
    # Source: https://www.kaggle.com/models/ipythonx/vsd-model/Keras/transunet
    model_dir = "/kaggle/input/vsd-model/keras/transunet/3"
    
    # --- Primary Model (Best: LB 0.545) ---
    # Trained with combo loss (Dice + CE), outputs raw logits (3 classes)
    input_shape = (160, 160, 160)
    num_classes = 3
    classifier_activation = None  # Raw logits → average before argmax
    weight_path = f"{model_dir}/transunet.seresnext50.160px.comboloss.weights.h5"
    
    # --- Sliding Window Inference ---
    overlap = 0.40  # 40% overlap (tuned: lower than 0.5 gives better results)
    sw_batch_size = 1
    sw_mode = 'gaussian'
    
    # --- Post-processing (optimized parameters from best submission) ---
    T_low = 0.30       # Lower hysteresis threshold
    T_high = 0.80      # Upper hysteresis threshold
    z_radius = 3       # Z-axis closing radius (aggressive: fills gaps along scroll layers)
    xy_radius = 2      # XY-plane closing radius (connects nearby sheet fragments)
    dust_min_size = 100  # Min connected component size
    
    # --- TTA ---
    use_tta = True
    
    # --- Seed ---
    seed = 42
    
    # --- Alternative model configs (for experimentation) ---
    alt_models = {
        "transunet_160_softmax": {
            "input_shape": (160, 160, 160),
            "num_classes": 3,
            "classifier_activation": "softmax",
            "weight_path": f"{model_dir}/transunet.seresnext50.160px.weights.h5",
            "lb_score": 0.505,
        },
        "transunet_128_softmax": {
            "input_shape": (128, 128, 128),
            "num_classes": 2,
            "classifier_activation": "softmax",
            "weight_path": f"{model_dir}/transunet.seresnext50.128px.weights.h5",
            "lb_score": 0.500,
        },
    }

os.makedirs(CFG.output_dir, exist_ok=True)
print("Configuration loaded.")
print(f"Primary model: TransUNet-160px-comboloss (logit output)")
print(f"  Weight path: {CFG.weight_path}")
print(f"Overlap: {CFG.overlap}")
print(f"Post-processing: T_low={CFG.T_low}, T_high={CFG.T_high}, z={CFG.z_radius}, xy={CFG.xy_radius}")
print(f"TTA: {'Enabled (8 views)' if CFG.use_tta else 'Disabled'}")

## 2. Data Loading & EDA

Load test data and examine the volume structure.  
- Competition data: 3D TIFF volumes, typically 256x256x256 voxels
- Labels: 0=background, 1=foreground (scroll surface), 2=unlabeled (ignore)
- Label thickness: ~3 voxels (updated dataset)

In [ ]:
# Load test metadata
test_df = pd.read_csv(f"{CFG.root_dir}/test.csv")
print(f"Test samples: {len(test_df)}")
print(f"Columns: {list(test_df.columns)}")
display(test_df.head())

# Check train data statistics if available
train_csv = f"{CFG.root_dir}/train.csv"
if os.path.exists(train_csv):
    train_df = pd.read_csv(train_csv)
    print(f"\nTraining samples: {len(train_df)}")
    print(f"Unique scroll_ids: {train_df['scroll_id'].nunique()}")
    print(f"\nScroll ID distribution:")
    print(train_df['scroll_id'].value_counts())

In [ ]:
# Examine a test volume
sample_id = test_df['id'].iloc[0]
sample_path = f"{CFG.test_dir}/{sample_id}.tif"

if os.path.exists(sample_path):
    vol = tifffile.imread(sample_path)
    print(f"Test volume {sample_id}:")
    print(f"  Shape: {vol.shape}")
    print(f"  Dtype: {vol.dtype}")
    print(f"  Value range: [{vol.min()}, {vol.max()}]")
    print(f"  Mean: {vol.mean():.2f}, Std: {vol.std():.2f}")
    print(f"  Non-zero fraction: {(vol > 0).mean():.4f}")
    
    # Visualize middle slices across 3 axes
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    d, h, w = vol.shape
    
    axes[0].imshow(vol[d//2], cmap='gray')
    axes[0].set_title(f'Axial (Z={d//2})')
    axes[0].axis('off')
    
    axes[1].imshow(vol[:, h//2, :], cmap='gray')
    axes[1].set_title(f'Coronal (Y={h//2})')
    axes[1].axis('off')
    
    axes[2].imshow(vol[:, :, w//2], cmap='gray')
    axes[2].set_title(f'Sagittal (X={w//2})')
    axes[2].axis('off')
    
    plt.suptitle(f'Test Volume {sample_id} - Middle Slices', fontsize=14)
    plt.tight_layout()
    plt.show()
    
    del vol
else:
    print(f"Test volume not found at {sample_path}")

## 3. Preprocessing Pipeline

Intensity normalization using z-score on nonzero voxels.  
This is the same normalization used during training for all TransUNet models.

In [ ]:
def load_volume(path):
    """Load a 3D TIFF volume and prepare for model input.
    Returns: np.ndarray of shape (1, D, H, W, 1) in float32.
    """
    vol = tifffile.imread(path)
    vol = vol.astype(np.float32)
    vol = vol[None, ..., None]  # Add batch and channel dims
    return vol


def val_transformation(image):
    """Apply z-score normalization on nonzero voxels.
    This matches the training-time normalization for TransUNet models.
    """
    data = {"image": image}
    pipeline = Compose([
        NormalizeIntensity(
            keys=["image"],
            nonzero=True,
            channel_wise=False
        ),
    ])
    result = pipeline(data)
    return result["image"]


print("Preprocessing functions defined.")

## 4. Model Loading

Load the best-performing TransUNet model with SEResNeXt50 encoder and Vision Transformer bottleneck.

### Model Selection Rationale

**Why single model over ensemble?**  
Experiments showed that the single comboloss model (LB 0.545) with optimized post-processing  
outperforms multi-model ensembles. This is because:

1. The comboloss model already captures strong topology through its Dice+CE training objective
2. Weaker models (LB 0.500-0.505) introduce noise that degrades thin-sheet predictions
3. The aggressive morphological closing (z=3, xy=2) compensates for any missing diversity

**Why `classifier_activation=None` (logits)?**  
Raw logit output enables TTA averaging in logit space before argmax, which preserves  
sharper class boundaries compared to averaging post-softmax probabilities.

### Architecture: TransUNet-SEResNeXt50
- **Encoder**: SEResNeXt50 (3D) with squeeze-and-excitation blocks
- **Bottleneck**: 12-layer Vision Transformer (embed_dim=512, 8 heads)
- **Decoder**: 5-level UNet decoder with LeakyReLU activation
- **Parameters**: ~70M
- **Output**: 3 classes (0=background, 1=foreground/surface, 2=unlabeled)

In [ ]:
# Load the primary model
print(f"Loading TransUNet-SEResNeXt50 (comboloss)...")
print(f"  Weight path: {CFG.weight_path}")

model = TransUNet(
    input_shape=(*CFG.input_shape, 1),
    encoder_name='seresnext50',
    classifier_activation=CFG.classifier_activation,
    num_classes=CFG.num_classes,
)
model.load_weights(CFG.weight_path)

params_m = model.count_params() / 1e6
print(f"  Parameters: {params_m:.1f}M")
print(f"  Output: {CFG.num_classes} classes, activation={CFG.classifier_activation}")

# Setup Sliding Window Inference
swi = SlidingWindowInference(
    model,
    num_classes=CFG.num_classes,
    roi_size=CFG.input_shape,
    sw_batch_size=CFG.sw_batch_size,
    mode=CFG.sw_mode,
    overlap=CFG.overlap,
)
print(f"\nSliding Window Inference configured:")
print(f"  ROI size: {CFG.input_shape}")
print(f"  Overlap: {CFG.overlap}")
print(f"  Mode: {CFG.sw_mode}")

# Print model architecture summary
model.instance_describe()

## 5. Test-Time Augmentation (TTA)

Apply 7 geometric augmentations during inference (+ original = 8 views total):

1. **3 flips** along each spatial axis (D, H, W)
2. **3 rotations** (90°, 180°, 270°) in the H-W plane

### TTA Strategy: Logit Averaging → Argmax

Since the model outputs raw logits (`classifier_activation=None`), we:
1. Collect logit tensors from all 8 augmented views
2. Average them in logit space (before any softmax)
3. Apply `argmax` to get hard class labels (0, 1, or 2)

This approach preserves sharper decision boundaries than averaging post-softmax probabilities,
because logits maintain relative class confidence magnitudes.

In [ ]:
def predict_with_tta(inputs, swi):
    """Run sliding window inference with TTA.
    
    Averages raw logits from 8 geometric augmentations, then takes argmax
    to produce hard class labels.
    
    Args:
        inputs: np.ndarray (1, D, H, W, 1) - preprocessed volume
        swi: SlidingWindowInference object
    
    Returns:
        class_map: np.ndarray (D, H, W) uint8 - class indices {0, 1, 2}
    """
    logits = []

    # Original
    logits.append(swi(inputs))

    if CFG.use_tta:
        # Flips along spatial axes (D=1, H=2, W=3)
        for axis in [1, 2, 3]:
            img_f = np.flip(inputs, axis=axis)
            p = swi(img_f)
            p = np.flip(p, axis=axis)
            logits.append(p)

        # Axial rotations in H-W plane
        for k in [1, 2, 3]:
            img_r = np.rot90(inputs, k=k, axes=(2, 3))
            p = swi(img_r)
            p = np.rot90(p, k=-k, axes=(2, 3))
            logits.append(p)

    # Average logits across all augmented views, then argmax
    mean_logits = np.mean(logits, axis=0)
    return mean_logits.argmax(-1).astype(np.uint8).squeeze()


print("TTA inference function defined.")
print(f"TTA enabled: {CFG.use_tta}")
n_views = 8 if CFG.use_tta else 1
print(f"Total inference passes per volume: {n_views}")

## 6. Post-Processing

### Three-stage post-processing pipeline:

1. **3D Hysteresis Thresholding**: Seeds strong predictions (≥ T_high) and propagates to weaker ones (≥ T_low).
   Since the input is argmax class indices {0, 1, 2}, the thresholds effectively select:
   - `T_high=0.80`: captures class 1 (foreground) and class 2 (unlabeled) — all non-background
   - `T_low=0.30`: same effective selection
   - 26-connectivity is used for propagation

2. **Anisotropic Morphological Closing**: Fills gaps in the scroll sheet predictions.
   - `z_radius=3`: aggressively connects along the z-axis (scroll layer direction)
   - `xy_radius=2`: moderately connects in the xy-plane
   - This is critical for maintaining continuous sheet topology

3. **Dust Removal**: Removes connected components smaller than 100 voxels.

### Why aggressive closing (z=3, xy=2)?
The updated dataset uses thin 3-voxel labels. The model predictions for these thin sheets
often have small gaps. Aggressive closing reconnects these fragments without merging
distinct sheets, because the sheets are spaced far apart in the updated labels.

In [ ]:
def build_anisotropic_struct(z_radius: int, xy_radius: int):
    """Build a 3D structuring element with different radii for z and xy.
    
    This allows morphological operations that respect the anisotropic
    nature of scroll surfaces (thin in z, extended in xy).
    """
    z, r = z_radius, xy_radius
    if z == 0 and r == 0:
        return None
    if z == 0 and r > 0:
        size = 2 * r + 1
        struct = np.zeros((1, size, size), dtype=bool)
        cy, cx = r, r
        for dy in range(-r, r + 1):
            for dx in range(-r, r + 1):
                if dy * dy + dx * dx <= r * r:
                    struct[0, cy + dy, cx + dx] = True
        return struct
    if z > 0 and r == 0:
        struct = np.zeros((2 * z + 1, 1, 1), dtype=bool)
        struct[:, 0, 0] = True
        return struct
    depth = 2 * z + 1
    size = 2 * r + 1
    struct = np.zeros((depth, size, size), dtype=bool)
    cz, cy, cx = z, r, r
    for dz in range(-z, z + 1):
        for dy in range(-r, r + 1):
            for dx in range(-r, r + 1):
                if dy * dy + dx * dx <= r * r:
                    struct[cz + dz, cy + dy, cx + dx] = True
    return struct


def topo_postprocess(
    class_map,
    T_low=0.30,
    T_high=0.80,
    z_radius=3,
    xy_radius=2,
    dust_min_size=100,
):
    """Topology-aware post-processing pipeline.
    
    Args:
        class_map: argmax output (D, H, W) with values {0, 1, 2}
        T_low: lower hysteresis threshold
        T_high: upper hysteresis threshold
        z_radius: closing radius along z-axis
        xy_radius: closing radius in xy-plane
        dust_min_size: minimum component size to keep
    
    Returns:
        mask: binary uint8 mask (D, H, W)
    """
    # Step 1: 3D Hysteresis Thresholding
    # For argmax output {0,1,2}: T_high=0.80 selects classes 1 and 2
    strong = class_map >= T_high
    weak = class_map >= T_low

    if not strong.any():
        return np.zeros_like(class_map, dtype=np.uint8)

    # 26-connectivity for propagation
    struct_hyst = ndi.generate_binary_structure(3, 3)
    mask = ndi.binary_propagation(
        strong, mask=weak, structure=struct_hyst
    )

    if not mask.any():
        return np.zeros_like(class_map, dtype=np.uint8)

    # Step 2: 3D Anisotropic Morphological Closing
    if z_radius > 0 or xy_radius > 0:
        struct_close = build_anisotropic_struct(z_radius, xy_radius)
        if struct_close is not None:
            mask = ndi.binary_closing(mask, structure=struct_close)

    # Step 3: Dust Removal
    if dust_min_size > 0:
        mask = remove_small_objects(
            mask.astype(bool), min_size=dust_min_size
        )

    return mask.astype(np.uint8)


# Visualize the structuring element
struct = build_anisotropic_struct(CFG.z_radius, CFG.xy_radius)
print(f"Structuring element shape: {struct.shape}")
print(f"Non-zero voxels: {struct.sum()}")
print(f"\nPost-processing parameters:")
print(f"  Hysteresis: T_low={CFG.T_low}, T_high={CFG.T_high}")
print(f"  Closing: z_radius={CFG.z_radius}, xy_radius={CFG.xy_radius}")
print(f"  Dust removal: min_size={CFG.dust_min_size}")

## 7. Full Inference Pipeline

Combine all components: loading → normalization → TTA prediction → post-processing.

The pipeline for each volume:
1. Load TIFF → float32 with batch/channel dims
2. Z-score normalize (nonzero voxels)
3. Sliding window inference × 8 TTA views → average logits → argmax
4. Hysteresis threshold → morphological closing → dust removal
5. Save binary uint8 mask

In [ ]:
def inference_pipeline(volume):
    """Complete inference pipeline for a single volume.
    
    Args:
        volume: preprocessed volume (1, D, H, W, 1)
    
    Returns:
        mask: binary prediction (D, H, W) uint8
        class_map: argmax class map (D, H, W) uint8, for diagnostics
    """
    # TTA prediction: logit averaging → argmax → class map {0, 1, 2}
    class_map = predict_with_tta(volume, swi)
    
    # Post-processing: hysteresis → closing → dust removal
    mask = topo_postprocess(
        class_map,
        T_low=CFG.T_low,
        T_high=CFG.T_high,
        z_radius=CFG.z_radius,
        xy_radius=CFG.xy_radius,
        dust_min_size=CFG.dust_min_size,
    )
    
    return mask, class_map


print("Full inference pipeline defined.")

## 8. Run Inference & Create Submission

Process all test volumes and create the submission ZIP file.

In [ ]:
# Store predictions for visualization
all_outputs = {}

with zipfile.ZipFile(
    CFG.zip_path, "w", compression=zipfile.ZIP_DEFLATED
) as z:
    for idx, image_id in enumerate(test_df["id"]):
        print(f"\n{'='*60}")
        print(f"Processing volume {idx+1}/{len(test_df)}: {image_id}")
        print(f"{'='*60}")
        
        tif_path = f"{CFG.test_dir}/{image_id}.tif"
        
        # Load and preprocess
        volume = load_volume(tif_path)
        print(f"  Raw shape: {volume.shape}")
        volume = val_transformation(volume)
        
        # Run inference pipeline
        output, class_map = inference_pipeline(volume)
        
        # Statistics
        fg_voxels = output.sum()
        total_voxels = output.size
        n_class1 = (class_map == 1).sum()
        n_class2 = (class_map == 2).sum()
        print(f"  Argmax classes: bg={total_voxels - n_class1 - n_class2:,}, fg={n_class1:,}, ign={n_class2:,}")
        print(f"  Final mask: {fg_voxels:,} / {total_voxels:,} ({fg_voxels/total_voxels*100:.2f}%)")
        
        # Save to zip
        out_path = f"{CFG.output_dir}/{image_id}.tif"
        tifffile.imwrite(out_path, output.astype(np.uint8))
        z.write(out_path, arcname=f"{image_id}.tif")
        os.remove(out_path)
        
        # Store for visualization
        all_outputs[image_id] = {
            "volume": volume,
            "mask": output,
            "class_map": class_map,
        }

print(f"\nSubmission ZIP saved: {CFG.zip_path}")

## 9. Visualization & Quality Check

Visualize predictions to verify quality before submission.

In [ ]:
def plot_prediction_slices(volume, mask, class_map, image_id, max_slices=5):
    """Plot input, argmax class map, and final binary mask side by side."""
    img = np.squeeze(volume)  # (D, H, W)
    D = img.shape[0]
    
    step = max(1, D // max_slices)
    slices = list(range(0, D, step))[:max_slices]
    n = len(slices)
    
    fig, axes = plt.subplots(3, n, figsize=(4*n, 12))
    
    for i, s in enumerate(slices):
        # Row 1: Input volume
        axes[0, i].imshow(img[s], cmap='gray')
        axes[0, i].set_title(f'Input (z={s})')
        axes[0, i].axis('off')
        
        # Row 2: Argmax class map (0=bg, 1=fg, 2=ignore)
        axes[1, i].imshow(class_map[s], cmap='hot', vmin=0, vmax=2)
        axes[1, i].set_title(f'Argmax (z={s})')
        axes[1, i].axis('off')
        
        # Row 3: Final binary mask
        axes[2, i].imshow(mask[s], cmap='gray')
        axes[2, i].set_title(f'Mask (z={s})')
        axes[2, i].axis('off')
    
    axes[0, 0].set_ylabel('Input', fontsize=12)
    axes[1, 0].set_ylabel('Argmax', fontsize=12)
    axes[2, 0].set_ylabel('Prediction', fontsize=12)
    
    plt.suptitle(f'Prediction for volume {image_id}', fontsize=14)
    plt.tight_layout()
    plt.show()


# Visualize all test predictions
for image_id, data in all_outputs.items():
    plot_prediction_slices(
        data["volume"],
        data["mask"],
        data["class_map"],
        image_id,
        max_slices=5
    )

In [ ]:
# 3-axis cross-section view
for image_id, data in all_outputs.items():
    mask = data["mask"]
    vol = np.squeeze(data["volume"])
    d, h, w = mask.shape
    
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    
    # Axial
    axes[0, 0].imshow(vol[d//2], cmap='gray')
    axes[0, 0].set_title(f'Axial (z={d//2}) - Input')
    axes[1, 0].imshow(mask[d//2], cmap='gray')
    axes[1, 0].set_title(f'Axial (z={d//2}) - Prediction')
    
    # Coronal
    axes[0, 1].imshow(vol[:, h//2, :], cmap='gray')
    axes[0, 1].set_title(f'Coronal (y={h//2}) - Input')
    axes[1, 1].imshow(mask[:, h//2, :], cmap='gray')
    axes[1, 1].set_title(f'Coronal (y={h//2}) - Prediction')
    
    # Sagittal
    axes[0, 2].imshow(vol[:, :, w//2], cmap='gray')
    axes[0, 2].set_title(f'Sagittal (x={w//2}) - Input')
    axes[1, 2].imshow(mask[:, :, w//2], cmap='gray')
    axes[1, 2].set_title(f'Sagittal (x={w//2}) - Prediction')
    
    for ax in axes.flat:
        ax.axis('off')
    
    plt.suptitle(f'Three-axis cross-section: {image_id}', fontsize=14)
    plt.tight_layout()
    plt.show()

## 10. Submission Verification

Verify the submission file is correctly formatted.

In [ ]:
# Verify submission
print("Submission verification:")
print(f"  ZIP path: {CFG.zip_path}")
print(f"  ZIP exists: {os.path.exists(CFG.zip_path)}")

if os.path.exists(CFG.zip_path):
    zip_size = os.path.getsize(CFG.zip_path)
    print(f"  ZIP size: {zip_size / 1024:.1f} KB")
    
    with zipfile.ZipFile(CFG.zip_path, 'r') as zf:
        file_list = zf.namelist()
        print(f"  Files in ZIP: {len(file_list)}")
        for f in file_list:
            info = zf.getinfo(f)
            print(f"    - {f} ({info.compress_size / 1024:.1f} KB compressed)")
    
    # Cross-check with test.csv
    expected_ids = set(test_df['id'].astype(str).tolist())
    actual_ids = set([f.replace('.tif', '') for f in file_list])
    
    if expected_ids == actual_ids:
        print("\n  All test IDs accounted for. Submission is valid.")
    else:
        missing = expected_ids - actual_ids
        extra = actual_ids - expected_ids
        if missing:
            print(f"\n  WARNING: Missing IDs: {missing}")
        if extra:
            print(f"\n  WARNING: Extra IDs: {extra}")

## Summary

### What this solution does:
1. **Single best model**: TransUNet-SEResNeXt50 with comboloss (LB 0.545)
2. **Logit-space TTA**: 8 geometric augmentations, averaged before argmax
3. **Optimized sliding window**: 40% overlap with Gaussian weighting
4. **Aggressive post-processing**: z=3, xy=2 morphological closing for thin-sheet connectivity

### Key hyperparameters (from best submission):
| Parameter | Value | Rationale |
|---|---|---|
| overlap | 0.40 | Lower overlap improves boundary consistency |
| T_low | 0.30 | Captures all non-background classes from argmax |
| T_high | 0.80 | Same effective selection on discrete {0,1,2} |
| z_radius | 3 | Fills z-direction gaps in thin 3vx labels |
| xy_radius | 2 | Connects nearby sheet fragments in xy |
| dust_min_size | 100 | Removes small noise components |

### Why this beats ensembles:
- Multi-model ensemble (2 models, weights 0.25:0.75) scored **worse** than single model
- Weaker models (LB 0.500-0.505) add noise to thin-sheet predictions
- Aggressive closing compensates for model diversity

### Possible improvements:
- Train on updated dataset with 3vx-thick labels
- Add topology-preserving loss functions (Betti matching, cl-Dice)
- Hole-filling via line tracing (as described by hengck23)
- Tune post-processing thresholds via local CV
- Avoid morphological closing entirely (host recommendation) and rely on cleaner model output